In [1]:
import polars as pl
import duckdb
from pathlib import Path
import plotly.express as px

In [2]:
con = duckdb.connect("trips.duckdb")
con.execute("SET memory_limit = '10GB'")

path = Path.cwd()
if not Path(path, "data").exists(): path = path.parent
path_yellow = path / "data" / "yellow_taxi_unified.parquet"
path_green = path / "data" / "green_taxi_unified.parquet"
path_uber = path / "data" / "uber_unified.parquet"
path_agg_data = path / "data" / "data_reports_monthly.csv"

In [31]:
color_map = {
    "Yellow": "#F2C200",
    "Green":  "#2E8B57",
    "FHV - High Volume": "#7B68EE",
}

In [ ]:
res1 = con.execute(f"""
    SELECT
    CAST(strptime("Month/Year" || '-01', '%Y-%m-%d') AS DATE) AS month_year_date,
    "Avg Hours Per Day Per Vehicle" as m_hours,
    "License Class" as type
    FROM '{path_agg_data}'
    WHERE "License Class" IN ('Yellow', 'Green', 'FHV - High Volume')    
    ORDER BY month_year_date DESC;

""").pl()

fig = px.line(
    res1,
    x="month_year_date",
    y="m_hours",
    color="type",
    title="Active hours per day per vehicle",
    color_discrete_map=color_map,
)

fig.update_xaxes(dtick="M12", tickformat="%Y")

fig.update_traces(
    hovertemplate="Date: %{x|%b %Y}<br>Mean Hours: %{y}<extra>%{fullData.name}</extra>"
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Mean Hours",
)

fig.show()

In [ ]:
# Trip-in-progress hours per day per vehicle
res2 = con.execute(f"""
    SELECT
    CAST(strptime("Month/Year" || '-01', '%Y-%m-%d') AS DATE) AS month_year_date,
    try_cast(replace("Unique Vehicles", ',', '') AS BIGINT) AS unique_vehicles,
    "License Class" as type
    FROM '{path_agg_data}'
    WHERE "License Class" IN ('Yellow', 'Green', 'FHV - High Volume')    
    ORDER BY month_year_date DESC;

""").pl()

fig = px.line(
    res2,
    x="month_year_date",
    y="unique_vehicles",
    color="type",
    title="Active hours per day per vehicle",
    color_discrete_map=color_map,
)

fig.update_xaxes(dtick="M12", tickformat="%Y")

fig.update_traces(
    hovertemplate="Date: %{x|%b %Y}<br>Unique Vehicles: %{y}<extra>%{fullData.name}</extra>"
)

fig.update_layout(
    xaxis_title="Date",
    yaxis_title="Unique Vehicles",
)

fig.show()